# Lab 3 - Solved: Generating tweets using n-grams

In [1]:
import os
import re
import emoji
import nltk
import pandas as pd
import kagglehub

from nltk.tokenize import word_tokenize
from nltk.tokenize.treebank import TreebankWordDetokenizer
from nltk.lm import MLE, Laplace
from nltk.lm.preprocessing import padded_everygram_pipeline, pad_both_ends
from nltk.util import ngrams
from sklearn.model_selection import train_test_split

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

C:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## 1. Load the data

In [2]:
path = kagglehub.dataset_download("adizafar/large-random-tweets-from-pakistan")
csv_file = [f for f in os.listdir(path) if f.endswith(".csv")][0]
df = pd.read_csv(os.path.join(path, csv_file), encoding="utf-8", encoding_errors="ignore", low_memory=False)
df.head()

,Unnamed: 0,created_at_tweet,full_text,retweet_count,favorite_count,reply_count,location
0,0,Wed Jul 03 07:20:15 +0000 2019,تیرا لیڈر میرا لیڈر نواز شریف نواز شریف❤️ کیا...,226,1097,127,"Punjab, Pakistan"
1,1,Fri Jul 02 02:50:53 +0000 2021,"Happy birthday to my brother n boss , May you ...",54,273,9,"Punjab, Pakistan"
2,2,Fri Jul 02 12:12:53 +0000 2021,❤️❤️,50,131,13,"Punjab, Pakistan"
3,3,Tue Oct 15 22:29:16 +0000 2019,`suspicious °jikook au jimin'in yaşadığı kasab...,522,2134,86,"İstanbul, Türkiye"
4,4,Wed May 13 04:34:31 +0000 2020,Speaking of @SpiderMan... 😂 https://t.co/uGJ...,74,535,11,Follow the ATP Tour ➡


## 2. Pre-processing

In [3]:
def clean_tweet(text):
    text = re.sub(r"#\w+", " ", text)
    text = re.sub(r"\bRT\b", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = emoji.replace_emoji(text, replace=" ")
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

clean = df['full_text'].dropna().astype(str).apply(clean_tweet)
clean = clean[clean.str.contains(r"[a-z]") & ~clean.str.contains(r"[^\x00-\x7F]")].drop_duplicates()

tokenized = [[t for t in word_tokenize(tweet) if re.search(r"[a-z0-9]", t)] for tweet in clean]
tokenized = [t for t in tokenized if len(t) >= 2]
print(len(tokenized))
print(tokenized[:3])

79907
[['happy', 'birthday', 'to', 'my', 'brother', 'n', 'boss', 'may', 'you', 'have', 'many', 'many', 'more', 'you', 'are', 'gem'], ['speaking', 'of'], ['alexa', 'skip', 'to', '2021']]


## 3. Build MLE model (bigram)

In [4]:
train_tweets, test_tweets = train_test_split(tokenized, test_size=0.1, random_state=42)

train_data, vocab = padded_everygram_pipeline(2, train_tweets)
lm = MLE(2)
lm.fit(train_data, vocab)
print(len(lm.vocab))

68732


In [5]:
detok = TreebankWordDetokenizer()

def generate_tweet(model, num_words=20, seed=None):
    words = []
    for w in model.generate(num_words, text_seed=["<s>"], random_seed=seed):
        if w == "</s>":
            break
        if w != "<s>":
            words.append(w)
    return detok.detokenize(words)

for i in range(10):
    print(generate_tweet(lm, seed=i))

this she is dirty politics discourse on genuine issues of this is every song it


bilkul sath nahany ma kasi he ook na


wo tou
excellent performance ends of preference


exactly ab karlain
our psl our voice we will also inadequate that pretty she always in everything is not
tention ni hr career 1 million ka pta ha or does not prepared for pakistan mei barkat market research attempted


help for social awerness is hopeful constructive role amp it's all now the cast did not unique peer pressure
ek village ameen lots love football you are removing our head lol


kaha hai


## 4. Evaluate the model

In [6]:
def to_bigrams(tweets):
    return [bg for t in tweets for bg in ngrams(pad_both_ends(t, n=2), 2)]

train_bigrams = to_bigrams(train_tweets[:1000])
test_bigrams = to_bigrams(test_tweets[:1000])

print("MLE perplexity (train):", lm.perplexity(train_bigrams))
print("MLE perplexity (test):", lm.perplexity(test_bigrams))

train_data, vocab = padded_everygram_pipeline(2, train_tweets)
laplace = Laplace(2)
laplace.fit(train_data, vocab)
print("Laplace perplexity (test):", laplace.perplexity(test_bigrams))

MLE perplexity (train): 79.7964654669944
MLE perplexity (test): inf


Laplace perplexity (test): 11062.512725055052


MLE gives unseen bigrams probability 0, so its test perplexity is infinite; Laplace smoothing fixes this.

## 5. Probability of the bigram (pakistan is)

In [7]:
print(lm.score("is", ["pakistan"]))

0.044846577498033044


## 6. Perplexity of the word (pakistan)

In [8]:
print(lm.perplexity([("pakistan",)]))

197.60503540519275
